# ThemeCloner
**Find undiscovered thematic exposure in small caps using RP-PCA.**

## Architecture

| Component | Role | Data source |
|-----------|------|-------------|
| ETF returns | Theme definition + OOS validation | yfinance (BOTZ, ICLN, AINF.L etc.) |
| Covariance universe | RP-PCA factor extraction | ACWI proxy: SP500+SP400+SP600+STOXX600+Nikkei225 |
| Target universe | Where we hunt for alpha | Russell 2000 proxy (NASDAQ+NYSE ex-SP500) |

## Data limitations (documented for paper)
- MSCI ACWI constituents require vendor licence → proxied with free index lists
- ETF constituent holdings blocked by providers → ETF price series used as theme labels
- Russell 2000 full list requires FTSE Russell licence → NASDAQ+NYSE ex-SP500 proxy

Edit `config/etfs.csv` to change themes and ETFs.

In [ ]:
# -------------------- 0. Setup --------------------

import sys, os

# make sure we run from the ThemeCloner root regardless of where Jupyter launched
ROOT = os.path.abspath(os.path.join(os.getcwd()))
if not os.path.exists(os.path.join(ROOT, 'config', 'etfs.csv')):
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(ROOT)
sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

print(f'root: {ROOT}')
print(f'config exists: {os.path.exists("config/etfs.csv")}')
print('setup done')

## Step 1 — Pull Data

In [ ]:
# -------------------- 1a. Load ETF config --------------------

from src.data_pull import (load_etf_config, pull_etf_returns,
                            pull_covariance_universe, pull_target_universe)

# parameters -- tune these for the run
START_DATE  = '2018-01-01'
K           = 10     # factors for covariance universe RP-PCA (more themes need more K)
GAMMA       = 10.0   # Lettau-Pelger default
TOP_N       = 30     # candidates per theme
MIN_SCORE   = 0.40   # minimum cosine similarity to be a candidate
TOP_FACTORS = 3      # how many factors per ETF to use for theme fingerprint

etf_config = load_etf_config('config/etfs.csv')
etf_config

In [ ]:
# -------------------- 1b. Pull ETF returns --------------------
# these serve two roles: theme definition (labeling factors) and OOS validation

etf_returns = pull_etf_returns(etf_config, start=START_DATE)
etf_returns.tail(3)

In [ ]:
# -------------------- 1c. Pull covariance universe (ACWI proxy) --------------------
# this is what RP-PCA runs on -- needs to be broad enough to capture all themes
# SP500 + SP400 + SP600 + STOXX600 + Nikkei225
# first run: ~10-15 min to pull. subsequent runs use cache.

cov_returns = pull_covariance_universe(start=START_DATE, use_cache=True)
print(f'covariance universe: {cov_returns.shape[1]} stocks x {cov_returns.shape[0]} weeks')

In [ ]:
# -------------------- 1d. Pull target universe (Russell 2000 proxy) --------------------
# small caps we score for undiscovered thematic exposure
# first run: ~20-30 min. subsequent runs use cache.

tgt_returns = pull_target_universe(start=START_DATE, use_cache=True)
print(f'target universe: {tgt_returns.shape[1]} stocks x {tgt_returns.shape[0]} weeks')

## Step 2 — Fit RP-PCA on Covariance Universe

In [ ]:
# -------------------- 2. RP-PCA on covariance universe --------------------
# K=10 to capture multiple themes as distinct latent factors
# the factor space is defined here -- everything else uses this same space

from src.rppca import fit_rppca, sweep_gamma

rppca_result = fit_rppca(cov_returns, K=K, gamma=GAMMA, run_oos=True)
print(f"\nloadings: {rppca_result['loadings'].shape}")
print(f"factors:  {rppca_result['factors'].shape}")

In [ ]:
# -------------------- gamma sweep --------------------
# confirm gamma=10 is reasonable for this universe

sweep = sweep_gamma(cov_returns.fillna(0).values, K=K)

fig, ax = plt.subplots()
ax.plot(sweep['gamma'], sweep['sr'], marker='o', color='steelblue', linewidth=2)
ax.axvline(GAMMA, color='firebrick', linestyle='--', label=f'gamma={GAMMA}')
ax.set_xlabel('gamma')
ax.set_ylabel('annualised Sharpe ratio')
ax.set_title(f'SR vs gamma  (K={K}, covariance universe)')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/fig_sr_vs_gamma.png', dpi=150)
plt.show()

In [ ]:
# -------------------- eigenvalue plot --------------------
# check how many factors are genuinely distinct (look for the elbow)

eigvals = rppca_result['eigvals']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(1, len(eigvals)+1), eigvals, color='steelblue')
axes[0].set_title('eigenvalues (RP-PCA, covariance universe)')
axes[0].set_xlabel('factor')

diffs = np.diff(eigvals)
axes[1].bar(range(1, len(diffs)+1), np.abs(diffs), color='steelblue')
axes[1].set_title('eigenvalue differences (elbow = natural K)')
axes[1].set_xlabel('factor')

plt.tight_layout()
plt.savefig('outputs/fig_eigenvalues.png', dpi=150)
plt.show()

## Step 3 — Label Theme Factors Using ETF Returns

In [ ]:
# -------------------- 3. Theme DNA --------------------
# regress each ETF on the K factors to identify which factors = which themes
# R² tells us how well the factor space captures each theme
# purity tells us how much the ETFs within a theme agree

from src.theme_dna import label_theme_factors

dna_result = label_theme_factors(
    rppca_result, etf_returns, etf_config,
    top_factors=TOP_FACTORS
)

print('\ntheme factor R² (how well factors explain each ETF):')
for etf, r2 in dna_result['etf_r2'].items():
    bar = '█' * int(r2 * 20)
    print(f'  {etf:20s}  R²={r2:.3f}  {bar}')

print('\ntheme purity (ETF agreement within theme):')
for theme, p in dna_result['purity'].items():
    bar = '█' * int(p * 20)
    print(f'  {theme:25s}  {p:.3f}  {bar}')

## Step 4 — Project Target Universe + Score

In [ ]:
# -------------------- 4. Project and score --------------------
# project each Russell proxy stock into the covariance factor space (OLS)
# score by cosine similarity to each theme fingerprint

from src.projection import project_universe, score_universe, save_projections

projections = project_universe(tgt_returns, rppca_result)
scores      = score_universe(projections, dna_result['theme_factors'])
save_projections(projections, scores)

# score distribution per theme
scores.describe()

In [ ]:
# -------------------- score distribution plot --------------------
# a healthy distribution is roughly normal, centered near 0
# the right tail (high scorers) are our candidates

fig, axes = plt.subplots(1, len(scores.columns), figsize=(5*len(scores.columns), 4))
if len(scores.columns) == 1:
    axes = [axes]

for ax, theme in zip(axes, scores.columns):
    s = scores[theme].dropna()
    ax.hist(s, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(MIN_SCORE, color='firebrick', linestyle='--',
               label=f'min_score={MIN_SCORE}')
    ax.set_title(theme)
    ax.set_xlabel('cosine similarity')
    ax.legend(fontsize=8)
    n_above = (s >= MIN_SCORE).sum()
    ax.text(0.98, 0.95, f'n>{MIN_SCORE}: {n_above}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9)

plt.suptitle('Score distribution (target universe vs theme factors)', y=1.02)
plt.tight_layout()
plt.savefig('outputs/fig_score_distribution.png', dpi=150)
plt.show()

## Step 5 — Rank Candidates

In [ ]:
# -------------------- 5. Rank candidates --------------------

from src.scoring import rank_candidates, save_outputs

ranked = rank_candidates(scores, top_n=TOP_N, min_score=MIN_SCORE)
save_outputs(ranked)

for theme in ranked['theme'].unique():
    print(f'\n--- {theme} (top 10) ---')
    print(ranked[ranked['theme'] == theme].head(10).to_string(index=False))

## Step 6 — OOS Validation Against ETF

In [ ]:
# -------------------- 6. OOS validation --------------------
# do our top candidates actually co-move with the theme ETF?
# this is the key validation -- no constituent data needed
# high ETF correlation = RP-PCA found genuine thematic exposure

from src.projection import validate_against_etf

validation = validate_against_etf(
    scores, tgt_returns, etf_returns, etf_config,
    top_n=TOP_N, min_score=MIN_SCORE
)
validation

In [ ]:
# -------------------- correlation plot --------------------
# for each theme: plot candidate portfolio return vs ETF return

for _, row in validation.iterrows():
    theme      = row['theme']
    candidates = row['candidates']
    theme_etfs = etf_config[etf_config['theme'] == theme]['ticker'].tolist()
    etf_avail  = [e for e in theme_etfs if e in etf_returns.columns]

    port_ret = tgt_returns[candidates].mean(axis=1)
    etf_ret  = etf_returns[etf_avail].mean(axis=1)
    common   = port_ret.index.intersection(etf_ret.index)

    # cumulative returns
    cum_port = (1 + port_ret.loc[common]).cumprod()
    cum_etf  = (1 + etf_ret.loc[common]).cumprod()

    fig, ax = plt.subplots()
    ax.plot(cum_port.index, cum_port.values, label='candidate portfolio', color='steelblue')
    ax.plot(cum_etf.index, cum_etf.values,   label='theme ETF',          color='firebrick', alpha=0.7)
    ax.set_title(f'{theme}  |  ETF corr = {row["etf_correlation"]:.3f}')
    ax.set_ylabel('cumulative return')
    ax.legend()
    plt.tight_layout()
    safe_name = theme.replace(' ', '_').replace('&', 'and')
    plt.savefig(f'outputs/fig_validation_{safe_name}.png', dpi=150)
    plt.show()